# Chapter 7. Semiempirical Quantum Chemistry

**Research question:** can a cheap electronic model remove obvious strain before a more expensive calculation, and how would we check the result? This chapter follows that decision on water: prepare a distorted structure, relax it with PM7, then evaluate both geometries with the same HF model. We also compare the methods' thermochemical predictions with a documented reference.

Semiempirical quantum methods reduce the cost of an electronic-structure calculation by simplifying its equations and introducing fitted parameters. The fitting data may be experimental measurements, higher-level calculations, or both. Electrons, orbitals, and a self-consistent electronic problem remain part of the model; these methods are distinct from the classical force fields of Chapter 2.

**Learning objectives**

- Distinguish an approximation family, a parameterized method, and a software implementation.
- Explain why parameter coverage and validation matter as much as computational speed.
- Run MOPAC single-point calculations and geometry optimizations with specified methods, charge, spin, and units.
- Interpret heats of formation, partial charges, and convergence diagnostics appropriately.
- Compare predictions with a cited reference without turning one successful example into a general accuracy claim.

**Prerequisites:** Chapters 4-6 and basic thermochemistry. The worked system is neutral, closed-shell water in the gas phase. The core examples run offline after installation and usually take seconds. They are calculations, not prerecorded results.

**Core route:** understand what is fitted, read one MOPAC input, examine charges and geometries, and follow the preoptimization decision. **Deeper reading:** NDDO integral notation, the method-family table, and the output-parser implementation. The parser is provided infrastructure: first understand the checks it makes, then revisit its regular expressions if useful.

**Working vocabulary:** *valence* electrons occupy the chemically active outer shells; *core* electrons are treated implicitly here. A *parameterization* is a collection of fitted constants used together with a specified model. *Transferability* asks whether a method works on chemistry outside its fitting examples. A *single point* solves the electronic problem while holding nuclei fixed.


## 7.1. What is approximated and what is fitted?

**Intuition:** fitted parameters can absorb some missing physics within the situations used to determine them. This saves work, but makes it especially important to test the property and chemical domain you need. An inexpensive calculation is still a model, not a lookup table of molecular answers.

Wavefunction methods in Chapter 5 evaluate molecular integrals for a chosen basis and then approximate the many-electron problem. Many traditional semiempirical models instead simplify or replace parts of those integrals, retain a small valence basis, and fit atom-specific parameters and core-repulsion terms. Their parameterized Hamiltonians still respond to the molecular environment through the electronic calculation.

The basis and parameters belong together. You cannot generally request `PM7/6-31G` in the way you request `HF/6-31G`: PM7's basis conventions are part of its parameterization. A minimal valence basis also does not mean one basis function per atom; an oxygen valence $s,p$ basis has four functions. Core electrons are treated implicitly in the MOPAC models used here.

### 7.1.1. Deeper reading: NDDO is a family of approximations

**NDDO** means *Neglect of Diatomic Differential Overlap*. It is not a separate parameter set to rank alongside AM1 or PM3. The labels $\mu,\nu,\lambda,\sigma$ below identify basis functions, not atoms or electrons. A two-electron integral describes Coulomb interactions between two pairs of basis functions. In a two-electron integral $(\mu\nu|\lambda\sigma)$, products of orbitals on different atomic centers are neglected within each orbital pair. Important one-center and two-center electron-interaction integrals remain. Additional approximations and fitted parameters give individual NDDO-based methods.

This does not mean that all interatomic interactions or all chemical bonding vanish. The treatment of one-electron terms, overlap, and core repulsion must be specified with the integral approximation. See the [MOPAC description of NDDO approximations](https://openmopac.net/Manual/approximations.html) and the [NDDO model review by Husch, Vaucher, and Reiher](https://arxiv.org/abs/1806.06147).

## 7.2. Method families and modern context (reference table)

| Method or family | What to remember |
| --- | --- |
| MNDO: Modified Neglect of Diatomic Overlap | A foundational NDDO-based parameterized method. Its name is distinct from the NDDO approximation. |
| AM1: Austin Model 1 | Modifies the MNDO core-repulsion treatment with additional Gaussian terms and its own fitted parameters. |
| PM3: Parametric Method 3 | Uses an AM1-like functional form with a different parameterization strategy and parameter values. |
| PM6 and PM7 | Later NDDO-based parameterizations with modified terms and broader element coverage. PM7 includes treatment of dispersion and hydrogen bonding within its model; it is not simply PM6 with a new label. |
| DFTB: density-functional tight binding | An approximate expansion related to DFT with its own parameter sets, often including a fitted repulsive contribution. It is not an NDDO method. |
| GFN1-xTB / GFN2-xTB | Parameterized extended tight-binding quantum methods, useful for rapid structural and noncovalent-interaction work within their validated scope. GFN2-xTB includes multipole electrostatics and a density-dependent dispersion contribution. |

Names such as **PM6-D3H4** identify particular corrected models; correction terms cannot be added indiscriminately to another parameterization. Likewise, **GFN-FF is a force field**, whereas GFN2-xTB is a quantum tight-binding method.

Useful primary sources: [MOPAC method bibliography](https://openmopac.net/Manual/bibliography.html), [PM7 formulation and parameterization](https://doi.org/10.1007/s00894-012-1667-x), [DFTB formulation](https://doi.org/10.1103/PhysRevB.58.7260), and [GFN2-xTB](https://doi.org/10.1021/acs.jctc.8b01176). These are different approximations, not a universal ladder in which the newest name always wins.

## 7.3. Choosing and evaluating a method

Semiempirical models often make geometry searches and screening substantially cheaper than larger-basis HF, correlated wavefunction, or DFT calculations. The actual speed ratio depends on the system, software, requested property, and convergence; there is no single universal factor.

Their limitations include parameter coverage, limited basis flexibility, approximate electron interactions, and errors outside the fitted chemical domain. Elements having parameters is necessary but insufficient: unusual oxidation states, radicals, charge transfer, transition-metal coordination, and noncovalent interactions need specific evidence. Solvation and electronic state are additional modeling choices.

Use a representative validation set and the observable you actually need. Compare errors, failure rates, and runtime under consistent settings. A higher-level method can be a useful reference, but it also has basis and methodological errors. Agreement on a molecule that contributed to parameter fitting is not an independent test of transferability.

**Keep three checks separate:** Did the electronic SCF converge? Did the geometry optimizer converge? Does the converged model describe the relevant chemistry accurately?

| Research decision | Evidence to collect | What one calculation cannot establish |
|---|---|---|
| Preoptimize structures before a chosen target method | Check chemical identity and the target method's energy/gradient on the resulting geometry | That the cheap optimum is also a target-method minimum |
| Rank candidate conformers | Compare representative relative energies against the intended reference method | A universal error bound from one molecule |
| Predict a thermochemical quantity | Match reaction, temperature, phase, and reference-state definitions | Accuracy from the most negative raw energy |

We execute the first and third checks on small systems. The table is a decision guide, not a numerical performance ranking.


## 7.4. MOPAC setup and input

[MOPAC](https://github.com/openmopac/mopac) is an implementation of several semiempirical methods. The current open-source distribution uses the Apache 2.0 license. This course uses MOPAC 23.2.5; the method is always specified explicitly.

Use the repository's updated `environment.yml` and select that environment as the Jupyter kernel. For an existing compatible Conda environment, the additional engine can be installed in a **terminal**:

```sh
conda install -c conda-forge mopac=23.2.5
```

A standalone [MOPAC download](https://openmopac.net/download/) is another option; preserve its bundled libraries. The code finds MOPAC in the current environment or on `PATH`. To use another installation, set the `MOPAC_EXECUTABLE` environment variable to its executable path before launching Jupyter. Missing software raises an error rather than silently substituting example answers.

In [ ]:
from pathlib import Path
import os
os.environ.setdefault("MKL_THREADING_LAYER", "SEQUENTIAL")
import re
import shutil
import subprocess
import sys
import tempfile

import numpy as np
import pandas as pd
from IPython import get_ipython
if get_ipython() is not None:
    get_ipython().run_line_magic("matplotlib", "inline")
else:
    import matplotlib
    matplotlib.use("Agg")
import matplotlib.pyplot as plt
from IPython.display import display

OUTPUT_DIR = Path("outputs/chapter07")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

override = os.environ.get("MOPAC_EXECUTABLE")
if override:
    candidates = [Path(override).expanduser()]
else:
    candidates = [Path(sys.prefix) / "Library/bin/mopac.exe",
                  Path(sys.prefix) / "bin/mopac"]
    if shutil.which("mopac"):
        candidates.append(Path(shutil.which("mopac")))
MOPAC = next((path.resolve() for path in candidates if path.is_file()), None)
if MOPAC is None:
    raise FileNotFoundError("MOPAC is missing. Install the course environment or set MOPAC_EXECUTABLE.")
print("MOPAC executable:", MOPAC.name)
print("Python:", sys.version.split()[0], "NumPy:", np.__version__)

### 7.4.1. A single point is not one SCF iteration

MOPAC input has a keyword line, **two comment lines**, then atom coordinates. We use Cartesian coordinates in angstroms. The `0` or `1` after each coordinate is an optimization flag. The example explicitly supplies `CHARGE=0 MS=0` for a neutral closed-shell state. More complicated spin states require additional choices; an $M_S$ value alone is not generally a proof of spin purity.

- `1SCF` requests one converged electronic calculation at a fixed geometry; it can require many electronic iterations.
- `GRADIENTS` requests derivatives even when `1SCF` is present.
- `PRECISE` tightens numerical criteria; it does not improve the underlying physical model.
- `AUX(PRECISION=6)` writes a machine-readable auxiliary file with extra printed digits.
- Removing `1SCF` and enabling coordinate flags requests optimization. `GNORM=0.01` sets a gradient-norm stopping threshold in kcal/(mol angstrom).

See the [input guide](https://openmopac.net/guides/basics/), [`1SCF`](https://openmopac.net/Manual/one_scf.html), [`AUX`](https://openmopac.net/Manual/auxiliary.html), and [`GNORM`](https://openmopac.net/Manual/gnorm.html). We retain input and output files in a fresh subfolder on each call, so a failed job cannot accidentally reuse an old result.

In [ ]:
METHODS = ("MNDO", "AM1", "PM3", "PM6", "PM7")
ELEMENTS = ("O", "H", "H")


def water_coordinates(bond_angstrom=0.9572, angle_deg=104.52):
    """A symmetric planar water geometry; these inputs are teaching choices."""
    theta = np.radians(angle_deg)
    return np.array([[0., 0., 0.], [bond_angstrom, 0., 0.],
                     [bond_angstrom*np.cos(theta), bond_angstrom*np.sin(theta), 0.]])


def water_input(method, xyz, optimize=False):
    if method not in METHODS:
        raise ValueError(f"Choose one of {METHODS}")
    xyz = np.asarray(xyz, dtype=float)
    if xyz.shape != (3, 3) or not np.isfinite(xyz).all():
        raise ValueError("Water coordinates must be a finite (3, 3) array.")
    operation = "GNORM=0.01" if optimize else "1SCF GRADIENTS"
    keywords = f"{method} {operation} CHARGE=0 MS=0 XYZ PRECISE AUX(PRECISION=6)"
    lines = [keywords, "Water classroom calculation", "Gas phase; Cartesian coordinates in angstroms"]
    flag = 1  # Keep coordinates eligible for derivatives; 1SCF prevents geometry optimization.
    for symbol, (x, y, z) in zip(ELEMENTS, xyz):
        lines.append(f"{symbol} {x:.10f} {flag} {y:.10f} {flag} {z:.10f} {flag}")
    return "\n".join(lines) + "\n\n"


fixed_xyz = water_coordinates()
print(water_input("PM7", fixed_xyz))

### Count what the model actually represents

All-electron neutral water has $8+1+1=10$ electrons. In this MOPAC model, the two oxygen $1s$ core electrons are implicit: oxygen contributes six valence electrons and each hydrogen contributes one, giving eight explicit valence electrons. Oxygen's one $s$ and three $p$ functions plus one $s$ function on each hydrogen give six basis functions. Electrons, basis functions, and atoms are different counts.

### Read the reported energy before using it

For gas-phase water, a standard formation enthalpy refers to the balanced reaction

$$\mathrm{H_2(g)}+\tfrac12\mathrm{O_2(g)}\longrightarrow\mathrm{H_2O(g)}.$$

The standard enthalpy of formation, $\Delta_fH^\circ$, is an energy change per mole formed from the elements in their reference states at a specified temperature. The traditional MOPAC models are parameterized to report formation enthalpies at 298.15 K. That reported quantity is **not** the bare clamped-nuclei total energy printed by an HF calculation. Unit conversion does not align their energy zeros or definitions. Do not add a zero-point correction to a MOPAC heat of formation as if it were an uncorrected HF electronic energy. See [MOPAC's thermochemical convention](https://openmopac.net/guides/basics/) and [energy components](https://openmopac.net/Manual/SCF_calc_hof.html).

### 7.4.2. Read results and check completion

The small helpers below read the labeled quantities needed for this lesson. MOPAC uses Fortran `D` exponents; for example, `-0.578D+02` means $-57.8$. An AUX array declares its length in brackets. Reading exactly that length helps catch truncated files.

The calculation wrapper checks the process status, normal termination, an explicit SCF-success message, the requested method, finite values, and coordinate/charge counts. For optimization it additionally checks the final gradient norm. These checks identify numerical and file failures; they do not establish chemical accuracy. Consult the saved `.out` file when a check fails. [MOPAC convergence criteria](https://openmopac.net/Manual/scf_test.html).

In [ ]:
def aux_scalar(text, key):
    matches = re.findall(r"^\s*" + re.escape(key) + r"=([^\r\n]+)", text, re.MULTILINE)
    if len(matches) != 1:
        raise ValueError(f"Expected exactly one AUX scalar {key}; found {len(matches)}")
    return matches[0].strip()


def aux_array(text, key):
    lines = text.splitlines()
    pattern = re.compile(r"^\s*" + re.escape(key) + r"\[(\d+)\]=(.*)$")
    for index, line in enumerate(lines):
        match = pattern.match(line)
        if match:
            count = int(match.group(1))
            values = match.group(2).split()
            while len(values) < count:
                index += 1
                if index >= len(lines):
                    raise ValueError(f"Truncated AUX array: {key}")
                if not lines[index].lstrip().startswith("#"):
                    values.extend(lines[index].split())
            if len(values) != count:
                raise ValueError(f"Wrong AUX array length: {key}")
            return np.array([float(value.replace("D", "E")) for value in values])
    raise KeyError(f"AUX array missing: {key}")


def run_water_job(method, xyz, optimize=False):
    run_dir = Path(tempfile.mkdtemp(prefix=method.lower()+"_", dir=OUTPUT_DIR))
    (run_dir / "water.mop").write_text(water_input(method, xyz, optimize), encoding="ascii")
    environment = os.environ.copy()
    environment["OMP_NUM_THREADS"] = "1"
    completed = subprocess.run([str(MOPAC), "water.mop"], cwd=run_dir,
                               env=environment, capture_output=True, text=True, timeout=10)
    (run_dir / "console.txt").write_text(completed.stdout + completed.stderr, encoding="utf-8")
    out_path, aux_path = run_dir / "water.out", run_dir / "water.aux"
    if completed.returncode != 0 or not out_path.is_file() or not aux_path.is_file():
        raise RuntimeError(f"MOPAC failed; inspect {run_dir}")
    output = out_path.read_text(encoding="utf-8", errors="replace")
    auxiliary = aux_path.read_text(encoding="utf-8", errors="replace")
    if ("JOB ENDED NORMALLY" not in output or "SCF FIELD WAS ACHIEVED" not in output
            or "END OF MOPAC PROGRAM" not in auxiliary):
        raise RuntimeError(f"MOPAC did not confirm a complete SCF calculation: {out_path}")
    if aux_scalar(auxiliary, "METHOD") != method:
        raise ValueError("Reported method differs from requested method.")
    def value(key):
        return float(aux_scalar(auxiliary, key).replace("D", "E"))
    result = {"method": method, "version": aux_scalar(auxiliary, "MOPAC_VERSION"),
              "heat_kcal_mol": value("HEAT_OF_FORMATION:KCAL/MOL"),
              "dipole_D": value("DIPOLE:DEBYE"),
              "gradient_norm": value("GRADIENT_NORM:KCAL/MOL/ANGSTROM"),
              "valence_electrons": int(aux_scalar(auxiliary, "NUM_ELECTRONS")),
              "coordinates": aux_array(auxiliary, "ATOM_X_OPT:ANGSTROMS").reshape(3, 3),
              "charges": aux_array(auxiliary, "ATOM_CHARGES"), "run_dir": run_dir}
    assert result["valence_electrons"] == 8  # O contributes 6 valence electrons; each H contributes 1.
    assert result["charges"].shape == (3,) and abs(result["charges"].sum()) < 1e-6
    assert np.isfinite([result[k] for k in ("heat_kcal_mol", "dipole_D", "gradient_norm")]).all()
    assert np.isfinite(result["coordinates"]).all() and np.isfinite(result["charges"]).all()
    if optimize and result["gradient_norm"] > 0.011:
        raise RuntimeError(f"Geometry gradient has not met GNORM=0.01; inspect {out_path}")
    if not optimize:
        np.testing.assert_allclose(result["coordinates"], xyz, atol=1e-8)
    return result

**Read the next output in this order:** method/version and successful termination; conserved charge and electron count; requested coordinate behavior; finite energy and gradient. The parser rejects incomplete results. A completed calculation still needs the scientific checks discussed above.

## 7.5. Compare methods at the same geometry

Keep coordinates, total charge, and spin treatment fixed so that this comparison changes the semiempirical method. The input dimensions are illustrative, not claimed to be an optimized geometry for every method. Every call performs a real electronic calculation.

In [ ]:
single_points = {method: run_water_job(method, fixed_xyz) for method in METHODS}
single_point_table = pd.DataFrame([
    {"method": method, "heat of formation (kcal/mol)": result["heat_kcal_mol"],
     "dipole (D)": result["dipole_D"],
     "gradient norm (kcal/mol/angstrom)": result["gradient_norm"]}
    for method, result in single_points.items()
]).set_index("method")
print("MOPAC versions:", sorted({r["version"] for r in single_points.values()}))
display(single_point_table.round(5))

### 7.5.1. What does the reported energy mean?

The MOPAC quantity used here is a **parameterized heat of formation**, conventionally associated with 298.15 K. Its construction includes method-dependent atomic reference terms. It is not the same energy zero as the total HF/DFT energies in Chapters 5-6, and it is not a Gibbs free energy. Converting units alone does not make these unlike quantities comparable. See the [heat-of-formation definition](https://openmopac.net/Manual/SCF_calc_hof.html).

At an unoptimized geometry this value is a model evaluation for that imposed structure. We optimize each method before making the thermochemical illustration below. The reported heat of formation already reflects the model's thermochemical parameterization; do not automatically add the zero-point and thermal corrections used in an ab initio thermochemistry recipe a second time.

Partial charges also depend on the electronic model and the population partition. They sum to the molecular charge, but individual atomic values are not uniquely measurable charges. A dipole is a distinct property of the whole charge distribution; a point-charge picture alone need not reproduce the program's dipole.

The next plot separates an atomic charge assignment from the molecular dipole. A molecular dipole is a property of the complete charge distribution; an atom's partial charge depends on how that distribution is partitioned. MOPAC's dipole also includes intra-atomic contributions, so reconstructing it from atomic point charges alone need not reproduce its output. Here we compare predictions at the same geometry, without asserting that either panel establishes accuracy. See the [MOPAC property definitions](https://openmopac.net/Manual/features.html).


In [ ]:
charge_table = pd.DataFrame({method: result["charges"] for method, result in single_points.items()},
                           index=["O", "H1", "H2"])
display(charge_table.round(5))
np.testing.assert_allclose(charge_table.sum(axis=0), 0.0, atol=1e-6)
print("Charge sums match the neutral molecule; this does not validate the partitioned atomic charges.")

fig, axes = plt.subplots(1, 2, figsize=(10, 3.5), layout="constrained")
axes[0].bar(METHODS, charge_table.loc["O"], color="#28788e")
axes[0].set(ylabel="Oxygen partial charge (units of e)", title="A model-dependent atomic partition")
axes[1].bar(METHODS, [single_points[m]["dipole_D"] for m in METHODS], color="#a96835")
axes[1].set(ylabel="Molecular dipole magnitude (D)", title="A whole-molecule property")
for axis in axes:
    axis.grid(axis="y", alpha=0.2)
    axis.set_axisbelow(True)
fig.savefig(OUTPUT_DIR / "charges_and_dipoles.png", dpi=150)
plt.show()


## 7.6. Optimize geometry and inspect the result

Start all methods from the same deliberately distorted water: two 1.15 angstrom O-H bonds and a 90-degree H-O-H angle. All Cartesian coordinates are free. Translation and rotation do not change internal geometry, so compare bond lengths and angles rather than raw atom positions.

The wrapper requires a small final gradient as well as SCF success. This is a converged geometry optimization, not a simulation at 298.15 K. A gradient close to zero indicates a stationary geometry; a Hessian/frequency calculation is needed to distinguish a minimum from a saddle point in general. This lesson does not perform that additional classification or claim a global minimum.

In [ ]:
distorted_xyz = water_coordinates(1.15, 90.0)
distorted_pm7 = run_water_job("PM7", distorted_xyz)
optimized = {method: run_water_job(method, distorted_xyz, optimize=True) for method in METHODS}


def water_geometry(xyz):
    vectors = np.asarray(xyz)[1:] - np.asarray(xyz)[0]
    lengths = np.linalg.norm(vectors, axis=1)
    cosine = np.dot(vectors[0], vectors[1]) / np.prod(lengths)
    return lengths, float(np.degrees(np.arccos(np.clip(cosine, -1., 1.))))


geometry_rows = []
for method, result in optimized.items():
    lengths, angle = water_geometry(result["coordinates"])
    geometry_rows.append({"method": method, "O-H1 (angstrom)": lengths[0],
        "O-H2 (angstrom)": lengths[1], "H-O-H (degrees)": angle,
        "heat of formation (kcal/mol)": result["heat_kcal_mol"],
        "gradient norm (kcal/mol/angstrom)": result["gradient_norm"]})
    assert result["heat_kcal_mol"] <= single_points[method]["heat_kcal_mol"] + 1e-5
geometry_table = pd.DataFrame(geometry_rows).set_index("method")
display(geometry_table.round(6))
assert optimized["PM7"]["heat_kcal_mol"] < distorted_pm7["heat_kcal_mol"]

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(9, 3.8), constrained_layout=True)
for ax, xyz, title in [(axes[0], distorted_xyz, "Distorted start"),
                        (axes[1], optimized["PM7"]["coordinates"], "PM7 optimized geometry")]:
    # Center on O for a display of this planar molecule; no coordinate modification in the result.
    centered = xyz - xyz[0]
    for hydrogen in (1, 2):
        ax.plot(centered[[0, hydrogen], 0], centered[[0, hydrogen], 1], color="0.5")
    for index, (symbol, point) in enumerate(zip(ELEMENTS, centered)):
        ax.scatter(point[0], point[1], s=100, color="#dc2626" if symbol == "O" else "#60a5fa")
        ax.annotate(f" {symbol}{index}", point[:2], xytext=(5, 5), textcoords="offset points")
    ax.set(xlabel="x (angstrom)", ylabel="y (angstrom)", title=title,
           xlim=(-0.7, 1.5), ylim=(-0.5, 1.5))
    ax.set_aspect("equal")
    ax.grid(alpha=0.2)
plt.show()

### A practical handoff: does PM7 preoptimization help the next model?

Cheap preoptimization is useful when it moves an obviously strained input toward a sensible starting structure for a target calculation. We can test that claim without optimizing twice. Evaluate the **same HF/STO-3G model** at (1) the distorted input and (2) the PM7-optimized geometry. A lower HF energy and smaller HF gradient would support using the second geometry as a starting guess for this example.

HF/STO-3G is a small teaching model, not an accuracy benchmark. We do not compare its total energy directly with the PM7 heat of formation. Both HF gradients use hartree per bohr, so their RMS values can be compared with each other; the MOPAC norm above has different units and a different norm convention.

For $N$ atoms and Cartesian gradient components $g_{A\alpha}=\partial E/\partial R_{A\alpha}$,

$$g_{\mathrm{RMS}}=\sqrt{\frac{1}{3N}\sum_{A=1}^{N}\sum_{\alpha\in\{x,y,z\}}g_{A\alpha}^{2}}.$$

Here $A$ labels atoms, $\alpha$ labels coordinate directions, and $R$ is a nuclear coordinate. A gradient measures how sensitive the energy remains to a small nuclear displacement. A small gradient is a stationarity test; curvature is still needed to classify a minimum. [Psi4 computes molecular gradients directly](https://psicode.org/psi4manual/1.11.x/opt.html), and [MOPAC describes its geometry optimization](https://openmopac.net/Manual/geometry_optimizer.html).

In [ ]:
import psi4

hf_scratch = (OUTPUT_DIR / "hf_scratch").resolve()
hf_scratch.mkdir(parents=True, exist_ok=True)
psi4.core.clean_options()
psi4.core.clean_variables()
psi4.core.IOManager.shared_object().set_default_path(str(hf_scratch))
psi4.core.set_output_file(str((OUTPUT_DIR / "hf_check.log").resolve()), False)
psi4.set_num_threads(1)
psi4.set_memory("512 MiB")
psi4.set_options({"basis": "sto-3g", "reference": "rhf", "scf_type": "pk",
                  "e_convergence": 1e-10, "d_convergence": 1e-9,
                  "maxiter": 100, "fail_on_maxiter": True})

hf_rows = []
for label, xyz in [("Distorted input", distorted_xyz),
                   ("After PM7 optimization", optimized["PM7"]["coordinates"])]:
    lines = ["0 1", *[f"{element} {x:.12f} {y:.12f} {z:.12f}"
                       for element, (x, y, z) in zip(ELEMENTS, xyz)],
             "units angstrom", "symmetry c1", "no_reorient", "no_com"]
    input_text = "\n".join(lines)
    molecule = psi4.geometry(input_text)
    gradient, wfn = psi4.gradient("hf", molecule=molecule, return_wfn=True)
    gradient_array = np.array(gradient.np, copy=True)
    assert gradient_array.shape == (3, 3) and np.isfinite(gradient_array).all()
    assert wfn.nalpha() == wfn.nbeta() == 5
    hf_rows.append({"geometry": label, "HF_energy_Eh": wfn.energy(),
                    "HF_gradient_RMS_Eh_per_bohr": np.sqrt(np.mean(gradient_array**2))})
    filename = "hf_distorted.in" if label == "Distorted input" else "hf_pm7_prepared.in"
    (OUTPUT_DIR / filename).write_text(input_text + "\n", encoding="utf-8")

handoff = pd.DataFrame(hf_rows).set_index("geometry")
handoff["HF_energy_change_kJ_mol"] = (handoff.HF_energy_Eh - handoff.HF_energy_Eh.iloc[0]) * psi4.constants.hartree2kJmol
assert np.isfinite(handoff.to_numpy()).all()
assert handoff.HF_energy_Eh.iloc[1] < handoff.HF_energy_Eh.iloc[0]
assert handoff.HF_gradient_RMS_Eh_per_bohr.iloc[1] < handoff.HF_gradient_RMS_Eh_per_bohr.iloc[0]
display(handoff.round(8))
handoff.to_csv(OUTPUT_DIR / "pm7_to_hf_geometry_check.csv")
psi4.core.clean()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 3.5), layout="constrained")
short_labels = ["Distorted input", "PM7 prepared"]
axes[0].bar(short_labels, handoff.HF_energy_change_kJ_mol, color=["#b8613d", "#28788e"])
axes[0].axhline(0, color="black", lw=1)
axes[0].set(ylabel="HF energy relative to distorted input (kJ/mol)",
            title="One target model, two geometries")
axes[1].bar(short_labels, handoff.HF_gradient_RMS_Eh_per_bohr, color=["#b8613d", "#28788e"])
axes[1].set(ylabel="HF Cartesian gradient RMS (hartree/bohr)",
            title="Smaller does not mean zero")
fig.savefig(OUTPUT_DIR / "pm7_to_hf_handoff.png", dpi=150)
plt.show()

**Decision supported here:** the PM7-prepared water is a less strained starting point for HF/STO-3G than the deliberately distorted input. Its nonzero HF gradient shows that optimizing under one model does not finish optimization under another. For real molecules, check bonding, stereochemistry, charge and spin as well; preoptimization can change the conformer or move to a different basin.

### 7.6.1. One thermochemical reference is a check, not a benchmark

For **gas-phase water**, the NIST Chemistry WebBook reports the CODATA review value

$$\Delta_f H^\circ_{\rm gas}(298.15\ \mathrm K)=-241.826\pm0.040\ \mathrm{kJ\,mol^{-1}}.$$

The phase matters: liquid water has a different formation enthalpy. The number is embedded here with its [NIST source](https://webbook.nist.gov/cgi/cbook.cgi?Name=water&cTC=on&cTG=on), so no network access is needed at execution time. We use the thermochemical conversion $1\ \mathrm{kcal}=4.184\ \mathrm{kJ}$.

We compare the optimized MOPAC estimates with this reference. The smallest error **for water in this illustration** does not identify the best general method. Water is a common parameterization/reference molecule, and we have not established an independent holdout test. Five models on one molecule do not give five independent experimental observations.

In [ ]:
reference_kJ_mol = -241.826
reference_uncertainty_kJ_mol = 0.040
comparison = pd.DataFrame({
    "method": METHODS,
    "predicted heat (kJ/mol)": [optimized[m]["heat_kcal_mol"] * 4.184 for m in METHODS],
}).set_index("method")
comparison["signed error (kJ/mol)"] = comparison["predicted heat (kJ/mol)"] - reference_kJ_mol
display(comparison.round(4))

fig, ax = plt.subplots(figsize=(7, 3.8), constrained_layout=True)
ax.bar(comparison.index, comparison["signed error (kJ/mol)"], color="#2563eb")
ax.axhline(0, color="black", linewidth=1)
ax.axhspan(-reference_uncertainty_kJ_mol, reference_uncertainty_kJ_mol,
           color="#f59e0b", alpha=0.5, label="NIST quoted uncertainty")
ax.set(ylabel="Prediction minus NIST reference (kJ/mol)",
       title="Optimized gas-phase water: one thermochemical example")
ax.legend(fontsize=9)
fig.savefig(OUTPUT_DIR / "water_formation_enthalpy_errors.png", dpi=150)
plt.show()

## 7.7. A controlled coordinate scan

For one method, scan the H-O-H angle while keeping both optimized O-H distances fixed at their mean. Each point is a single-point calculation on a separately constructed geometry. This is a **rigid angle scan**, not a relaxed scan, reaction path, or thermal distribution.

Subtract the minimum of this scan to display an energy difference within the same method and chemical composition. The atom-reference contributions to the heat of formation then cancel. This cancellation does not justify mixing raw energies from different methods.

In [ ]:
pm7_lengths, pm7_angle = water_geometry(optimized["PM7"]["coordinates"])
scan_angles = np.linspace(85.0, 125.0, 21)
scan_heats = np.array([run_water_job("PM7", water_coordinates(pm7_lengths.mean(), angle))["heat_kcal_mol"]
                       for angle in scan_angles])
relative_heats = (scan_heats - scan_heats.min()) * 4.184
assert np.isfinite(relative_heats).all() and np.min(relative_heats) == 0
assert abs(scan_angles[np.argmin(scan_heats)] - pm7_angle) < 6.0

fig, ax = plt.subplots(figsize=(7, 3.8), constrained_layout=True)
ax.plot(scan_angles, relative_heats, label="PM7 rigid scan")
ax.axvline(pm7_angle, color="0.4", linestyle="--", label="Optimized angle")
ax.set(xlabel="H-O-H angle (degrees)", ylabel="Energy above sampled minimum (kJ/mol)",
       title="Water angle scan at fixed O-H distances")
ax.legend()
plt.show()
print("All single-point, charge, geometry-convergence, and scan checks passed.")

## 7.8. Reporting and exercises

Report the program/version, full method name, geometry source, charge and spin treatment, solvent choice, calculation type, convergence settings, and energy definition/units. Preserve input and output files alongside the analysis. The `run_dir` fields above identify the saved jobs. For a real comparison, use a representative independent dataset, state its provenance, and separate parameter fitting from evaluation.

1. Explain why NDDO, PM7, and MOPAC name three different things. Why is `PM7/6-31G` generally inappropriate?
2. Water has ten electrons in an all-electron calculation. Why does the MOPAC AUX file report eight electrons and six valence basis functions here?
3. Why can an SCF-converged single point have a large nuclear gradient? Does `1SCF` mean a single SCF iteration?
4. Convert $-241.826$ kJ/mol to kcal/mol. What is the enthalpy for forming two moles of gaseous water from the elements in their standard reference states, using the NIST value? Why would liquid water require another reference?
5. Compare the fixed-geometry and optimized predictions. Separate changes due to the method from changes due to the geometry. Why is selecting the most negative value across different methods not an accuracy test?
6. Rerun the angle scan with half the spacing over the same interval. How does the estimated location of the sampled minimum change? Why does a rigid scan not give a finite-temperature angle distribution?
7. The PM7-prepared geometry has a smaller HF gradient than the distorted input. Why does that still not establish an HF minimum? Can you subtract its PM7 heat of formation from its HF energy after converting both to kJ/mol?
8. Design a validation study for hydrogen-bonded organic molecules. Specify reference properties, chemical coverage, independent test cases, failed jobs, and how you would summarize both error and cost.

<details><summary>Selected answers</summary>

1. NDDO is an integral-approximation family; PM7 is a parameterized model; MOPAC is software. PM7's basis conventions are part of its model rather than an independently selectable Gaussian basis.
2. The oxygen core is implicit. Oxygen contributes six valence electrons in four $s,p$ functions, and the two hydrogens contribute one electron and one function each.
3. SCF convergence adjusts the electronic solution at fixed nuclei. Nuclear stationarity is a separate condition. `1SCF` requests a completed single-point electronic calculation.
4. Approximately $-57.7978$ kcal/mol; formation of two moles gives $-483.652$ kJ. Enthalpy depends on the phase.
5. Each method approximates the system differently; accuracy requires a relevant reference, not simply a lower computed number. Optimize and compare using consistent definitions.

6. Finer spacing can resolve a sampled minimum more closely, but a rigid energy scan holds other coordinates fixed and has neither thermal populations nor their associated entropy.
7. The target-method gradient may remain nonzero, and internal curvature has not been classified. The quantities have different model definitions and energy references; converting units does not make their difference meaningful. Compare geometries using one consistent method and energy definition.

</details>

**Further reading:** [MOPAC open-source release](https://doi.org/10.21105/joss.08025), [MOPAC manual](https://openmopac.net/Manual/), and [GFN2-xTB original paper](https://doi.org/10.1021/acs.jctc.8b01176).

This completes the introductory quantum-chemistry sequence. Continue to [Chapter 8, Part 1](Chapter08_Part1.ipynb) to study computed molecular properties. Keep the model, numerical checks, and evidence for chemical accuracy separate.